# Barrier Certificates — Unified Monte Carlo Framework

A clean, vectorized research framework for pricing and risk-analyzing
**worst-of barrier certificates** on multiple underlyings.

The framework is fully **modular** and runs two complementary engines:

- **Risk-Neutral Monte Carlo** — drift = `r - q` → fair-value pricing
- **Real-World Monte Carlo** — drift = historical μ → risk metrics, scenarios

Both engines share the same:
- data layer (Excel/CSV ingestion, alignment, log-returns)
- volatility layer (historical + GARCH(1,1)-t)
- correlation structure (Cholesky on log-return correlations)
- payoff engine (worst-of, conditional coupon with optional memory, autocall, capital protection)

Run the cells top-to-bottom.

## 1. Configuration

Single point of truth. Edit **here only** to change the run.

In [ ]:
# %% CONFIGURATION
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from pathlib import Path

# ---------------------------------------------------------------- input data
# Put your historical price files (CSV or XLSX) in this folder.
# Each file must contain columns close/open/high/low/volume and a date column,
# OR a simple two-column (date, close) table. The loader is tolerant.
DATA_DIR = Path(".")

INPUT_FILES: Dict[str, str] = {
    "STOCK1": "INTESA.csv",
    "STOCK2": "BPM.csv",
    "STOCK3": "FINECO.csv",
    "STOCK4": "SA.csv",
}

# ---------------------------------------------------------------- market data
# Risk-free rate (annualized, continuous). Single value used for discounting
# and for the risk-neutral drift of every underlying.
RISK_FREE_RATE = 0.0221

# Forward dividend yield per ticker (annualized, continuous).
DIVIDEND_YIELD: Dict[str, float] = {
    "STOCK1": 0.0592,
    "STOCK2": 0.0815,
    "STOCK3": 0.0325,
    "STOCK4": 0.0222,
}

# ---------------------------------------------------------------- MC settings
N_DAYS    = 756       # simulation horizon (trading days). 252 = 1y, 504 = 2y, 756 = 3y
N_PATHS   = 20_000    # Monte Carlo paths
TAU       = 60        # GARCH mean-reversion speed for forward variance (in days)
SEED      = 42

# Distribution of innovations: "t" (heavy-tailed, recommended) or "normal"
INNOV_DIST = "t"

# ---------------------------------------------------------------- conventions
TRADING_DAYS = 252
ROLLING_WIN  = 30     # days, for rolling realized volatility


# ---------------------------------------------------------------- certificate
@dataclass
class CertificateSpec:
    """Specification of a worst-of barrier certificate."""
    notional: float = 100.0           # face value
    issue_price: float = 100.0        # issue/secondary price (for P&L baseline)
    barrier_pct: float = 0.40         # barrier as fraction of initial spot
    coupon_annual_pct: float = 7.80   # gross annual coupon, %
    coupon_freq_days: int = 30        # coupon observation frequency (~monthly)
    maturity_days: int = 504          # life of the product, in trading days

    # ---- optional features ----
    memory_coupon: bool = False       # if True, missed coupons accrue and are paid
                                      # the next time the barrier holds
    autocall: bool = False            # early redemption feature
    autocall_first_day: int = 252     # earliest autocall observation
    autocall_freq_days: int = 252     # spacing between autocall observations
    autocall_trigger_pct: float = 1.0 # worst-of >= trigger * S0 redeems at par + coupons

    # ---- capital protection at maturity ----
    capital_protection_pct: float = 0.0   # 0 = pure barrier, 1 = full protection
                                          # (e.g. 0.9 = 90% capital guaranteed)

CERTIFICATE = CertificateSpec()


# ---------------------------------------------------------------- output
OUTPUT_FILE = "Certificates_Report.xlsx"


## 2. Imports

In [ ]:
# %% IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import t as student_t, norm
from arch import arch_model
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="arch")

# Plot style — clean, presentation quality
plt.rcParams.update({
    "figure.figsize":   (10, 5),
    "figure.dpi":       110,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.25,
    "grid.linestyle":   "--",
    "axes.titleweight": "bold",
    "axes.titlesize":   12,
    "axes.labelsize":   10,
    "legend.frameon":   False,
    "font.size":        10,
})

PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd",
           "#ff7f0e", "#17becf", "#8c564b", "#e377c2"]

np.random.seed(SEED)
print("Environment ready.")


## 3. Data Loading

Loads CSV/XLSX with tolerance for messy real-world formats (Italian decimals,
volume suffixes, varying column orders), forward-fills small gaps, aligns
all underlyings on common trading days.

In [ ]:
# %% DATA LOADING
def _to_float(x):
    """Robust scalar parser: handles Italian comma, K/M/B suffixes."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("\u00a0", "").replace(" ", "")
    if s == "" or s.lower() in {"nan", "n/a", "-"}:
        return np.nan
    s = s.replace(",", ".")
    mult = 1.0
    if s[-1].upper() in "KMB":
        mult = {"K": 1e3, "M": 1e6, "B": 1e9}[s[-1].upper()]
        s = s[:-1]
    try:
        return float(s) * mult
    except ValueError:
        return np.nan


def load_price_file(path: Path) -> pd.DataFrame:
    """
    Load one historical price file (CSV or XLSX) and return a DataFrame
    indexed by date with a 'close' column (other OHLCV columns are kept
    when available but not required downstream).
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    # Normalize column names
    df.columns = [str(c).strip().lower() for c in df.columns]

    # Detect date column
    date_col = next((c for c in df.columns
                     if c in {"date", "data", "datetime"}), df.columns[0])
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce", dayfirst=False)
    if df[date_col].isna().any():
        # try EU dayfirst fallback
        df[date_col] = pd.to_datetime(df[date_col].astype(str),
                                      errors="coerce", dayfirst=True)
    df = df.dropna(subset=[date_col]).set_index(date_col).sort_index()

    # Detect close column
    close_col = next((c for c in df.columns
                      if c in {"close", "chiusura", "ultimo", "price"}),
                     df.columns[0])
    df["close"] = df[close_col].map(_to_float)

    # Optional OHLCV
    for c in ["open", "high", "low", "volume"]:
        if c in df.columns:
            df[c] = df[c].map(_to_float)

    keep = [c for c in ["open", "high", "low", "close", "volume"] if c in df.columns]
    df = df[keep]

    # Small gap fill (max 3 missing trading days)
    df = df.ffill(limit=3)

    if df["close"].isna().any():
        n = df["close"].isna().sum()
        print(f"   ! {path.name}: {n} residual NaN in close after ffill")
    return df


# ---- load every file, then align on common index --------------------------
raw_data = {tic: load_price_file(DATA_DIR / fname)
            for tic, fname in INPUT_FILES.items()}

prices_df = pd.concat([d["close"] for d in raw_data.values()],
                      axis=1, keys=raw_data.keys()).dropna()

TICKERS: List[str] = list(prices_df.columns)
N_ASSETS = len(TICKERS)

print(f"Loaded {N_ASSETS} underlyings: {TICKERS}")
print(f"Aligned history: {prices_df.index.min().date()} → "
      f"{prices_df.index.max().date()}  ({len(prices_df)} days)")
prices_df.tail()


## 4. Historical Analysis

Log-returns, realized volatility, correlation, VaR / Expected Shortfall,
drawdowns, and key performance statistics.

In [ ]:
# %% HISTORICAL ANALYSIS - returns
log_ret = np.log(prices_df / prices_df.shift(1)).dropna()
log_ret.columns = [f"{t}_rD" for t in TICKERS]

# rolling sums for multi-day horizons
returns = log_ret.copy()
for t in TICKERS:
    rd = returns[f"{t}_rD"]
    returns[f"{t}_rW"]  = rd.rolling(5).sum()
    returns[f"{t}_rM"]  = rd.rolling(21).sum()
    returns[f"{t}_rY"]  = rd.rolling(252).sum()
    returns[f"{t}_r3Y"] = rd.rolling(756).sum()

print("Daily log-returns + rolling cumulants computed.")
log_ret.tail()


In [ ]:
# %% HISTORICAL ANALYSIS - rolling volatility
rolling_vol = pd.DataFrame(index=log_ret.index)
for t in TICKERS:
    rd = log_ret[f"{t}_rD"]
    rolling_vol[f"{t}_vol_daily"]  = rd.rolling(ROLLING_WIN).std()
    rolling_vol[f"{t}_vol_annual"] = rolling_vol[f"{t}_vol_daily"] * np.sqrt(TRADING_DAYS)

# Full-sample annualized vols (one number per ticker, used as fallback)
hist_vol_annual = pd.Series({t: log_ret[f"{t}_rD"].std() * np.sqrt(TRADING_DAYS)
                             for t in TICKERS}, name="hist_vol_annual")
print("Full-sample annualized historical volatility:")
print(hist_vol_annual.round(4))


In [ ]:
# %% HISTORICAL ANALYSIS - correlation matrix
corr_matrix = log_ret.corr()
corr_matrix.index   = TICKERS
corr_matrix.columns = TICKERS
print("Correlation matrix of daily log-returns:")
print(corr_matrix.round(3))


In [ ]:
# %% HISTORICAL ANALYSIS - VaR, Expected Shortfall, drawdowns
def historical_var_es(returns_series: pd.Series, alpha: float = 0.95):
    """
    Historical 1-day VaR and ES at confidence `alpha` on the loss distribution.
    Returns positive numbers (loss magnitudes).
    """
    losses = -returns_series.dropna()
    var = losses.quantile(alpha)
    es  = losses[losses >= var].mean()
    return float(var), float(es)


def drawdown_stats(price_series: pd.Series) -> dict:
    cummax = price_series.cummax()
    dd = price_series / cummax - 1.0
    return {
        "max_drawdown":   float(dd.min()),
        "current_drawdown": float(dd.iloc[-1]),
        "avg_drawdown":   float(dd[dd < 0].mean()) if (dd < 0).any() else 0.0,
    }


def performance_stats(price_series: pd.Series) -> dict:
    r = np.log(price_series / price_series.shift(1)).dropna()
    n_years = len(r) / TRADING_DAYS
    cagr = (price_series.iloc[-1] / price_series.iloc[0]) ** (1 / n_years) - 1
    vol  = r.std() * np.sqrt(TRADING_DAYS)
    sharpe = (r.mean() * TRADING_DAYS - RISK_FREE_RATE) / vol if vol > 0 else np.nan
    return {
        "CAGR":           float(cagr),
        "Vol_annual":     float(vol),
        "Sharpe":         float(sharpe),
        "Skew":           float(r.skew()),
        "ExcessKurtosis": float(r.kurt()),
    }


# Aggregate per-ticker risk + performance dashboard
hist_summary = []
for t in TICKERS:
    row = {"Ticker": t, **performance_stats(prices_df[t])}
    var95, es95 = historical_var_es(log_ret[f"{t}_rD"], 0.95)
    var99, es99 = historical_var_es(log_ret[f"{t}_rD"], 0.99)
    row.update({"VaR_95": var95, "ES_95": es95,
                "VaR_99": var99, "ES_99": es99})
    row.update(drawdown_stats(prices_df[t]))
    hist_summary.append(row)

hist_summary_df = pd.DataFrame(hist_summary).set_index("Ticker")
print("Historical risk & performance summary:")
hist_summary_df.round(4)


### Plots — Prices, Returns, Volatility, Correlation

In [ ]:
# %% HISTORICAL PLOTS - prices and returns
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

# Normalized prices
norm_prices = prices_df / prices_df.iloc[0] * 100
for i, t in enumerate(TICKERS):
    axes[0].plot(norm_prices.index, norm_prices[t],
                 color=PALETTE[i], lw=1.4, label=t)
axes[0].set_title("Normalized Prices (base = 100)")
axes[0].set_ylabel("Index")
axes[0].legend(ncol=N_ASSETS, loc="upper left")

# Daily log-returns
for i, t in enumerate(TICKERS):
    axes[1].plot(log_ret.index, log_ret[f"{t}_rD"],
                 color=PALETTE[i], lw=0.6, alpha=0.7, label=t)
axes[1].set_title("Daily Log-Returns")
axes[1].set_ylabel("log-return")
axes[1].xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout(); plt.show()


In [ ]:
# %% HISTORICAL PLOTS - rolling volatility + return distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Rolling annualized vol
for i, t in enumerate(TICKERS):
    axes[0].plot(rolling_vol.index, rolling_vol[f"{t}_vol_annual"],
                 color=PALETTE[i], lw=1.3, label=t)
axes[0].set_title(f"Rolling {ROLLING_WIN}-day Annualized Volatility")
axes[0].set_ylabel("Vol (annual)")
axes[0].legend(ncol=N_ASSETS, fontsize=8)

# Return distributions
for i, t in enumerate(TICKERS):
    axes[1].hist(log_ret[f"{t}_rD"], bins=80, alpha=0.45,
                 color=PALETTE[i], label=t, density=True)
axes[1].set_title("Daily Log-Return Distributions")
axes[1].set_xlabel("log-return")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# %% HISTORICAL PLOTS - correlation heatmap
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(N_ASSETS)); ax.set_yticks(range(N_ASSETS))
ax.set_xticklabels(TICKERS, rotation=30, ha="right")
ax.set_yticklabels(TICKERS)
for i in range(N_ASSETS):
    for j in range(N_ASSETS):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", color="black", fontsize=9)
ax.set_title("Correlation — Daily Log-Returns")
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.04)
plt.tight_layout(); plt.show()


## 5. GARCH Volatility Model

GARCH(1,1) with Student-t innovations, fit on each ticker's log-returns.
The fit gives:

- the **current conditional volatility** σ₀ used as the starting point of
  the forward variance path,
- the **unconditional (long-run) variance** σ²_LR = ω / (1 − α − β),
- the **degrees of freedom** ν, used to draw heavy-tailed innovations
  during simulation.

The forward variance is mean-reverting toward σ²_LR with characteristic
time τ — see the engine cell.

In [ ]:
# %% GARCH MODELING
def fit_garch(returns_pct: pd.Series, p: int = 1, q: int = 1, dist: str = "t"):
    """
    Fit GARCH(p,q) on returns expressed in percent (arch convention).
    Returns (fit_result, params_dict). All params re-scaled to decimal.
    """
    am = arch_model(returns_pct, mean="Zero", vol="GARCH", p=p, q=q, dist=dist)
    res = am.fit(disp="off")
    P = res.params

    # arch fits on r * 100, so omega is in (pct)^2; rescale to decimal^2.
    out = {
        "omega": float(P["omega"]) / 1e4,
        "alpha": float(P[f"alpha[1]"]),
        "beta":  float(P[f"beta[1]"]),
        "nu":    float(P["nu"]) if "nu" in P.index else np.inf,
    }
    out["sigma2_lr"] = out["omega"] / max(1 - out["alpha"] - out["beta"], 1e-6)
    return res, out


garch_fits, garch_params = {}, {}
garch_vol_history = pd.DataFrame(index=log_ret.index)

for t in TICKERS:
    fit, p = fit_garch(log_ret[f"{t}_rD"].dropna() * 100, dist=INNOV_DIST)
    garch_fits[t]   = fit
    garch_params[t] = p
    # conditional volatility back to decimal scale
    cv_daily = fit.conditional_volatility / 100
    garch_vol_history[f"{t}_vol_daily"]  = cv_daily
    garch_vol_history[f"{t}_vol_annual"] = cv_daily * np.sqrt(TRADING_DAYS)

# Current σ0 = last conditional volatility
sigma0 = {t: float(garch_vol_history[f"{t}_vol_daily"].dropna().iloc[-1])
          for t in TICKERS}

print("GARCH(1,1)-{} fit summary:".format(INNOV_DIST))
g_summary = pd.DataFrame({
    "alpha":     [garch_params[t]["alpha"]      for t in TICKERS],
    "beta":      [garch_params[t]["beta"]       for t in TICKERS],
    "alpha+beta":[garch_params[t]["alpha"] + garch_params[t]["beta"] for t in TICKERS],
    "sigma_LR_ann":[np.sqrt(garch_params[t]["sigma2_lr"] * TRADING_DAYS) for t in TICKERS],
    "sigma0_ann":[sigma0[t] * np.sqrt(TRADING_DAYS) for t in TICKERS],
    "nu":        [garch_params[t]["nu"]         for t in TICKERS],
}, index=TICKERS)
g_summary.round(4)


In [ ]:
# %% GARCH PLOT - rolling vs GARCH conditional vol
fig, ax = plt.subplots(figsize=(11, 4.5))
for i, t in enumerate(TICKERS):
    ax.plot(rolling_vol.index, rolling_vol[f"{t}_vol_annual"],
            color=PALETTE[i], lw=0.9, alpha=0.55, label=f"{t} rolling")
    ax.plot(garch_vol_history.index, garch_vol_history[f"{t}_vol_annual"],
            color=PALETTE[i], lw=1.5, label=f"{t} GARCH")
ax.set_title("Annualized Volatility — Rolling vs GARCH(1,1) Conditional")
ax.set_ylabel("Vol (annual)")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.show()


## 6. Monte Carlo Engine

A single vectorized engine simulates correlated GBM paths for all
underlyings. The drift mode is the only difference between fair-value
pricing and risk analysis:

| Mode            | Daily drift                                     |
|-----------------|-------------------------------------------------|
| `risk_neutral`  | `(r − q − ½ σ²) · dt`                          |
| `real_world`    | historical mean log-return per asset (daily)   |

Innovations are correlated via Cholesky decomposition of the historical
log-return correlation matrix. Variance is GARCH-mean-reverting toward
σ²_LR with characteristic time τ.

In [ ]:
# %% MONTE CARLO ENGINE
def simulate_paths(
    S0: dict,
    sigma0: dict,
    garch_params: dict,
    corr_matrix: np.ndarray,
    *,
    mode: str = "risk_neutral",
    risk_free: float = 0.0,
    div_yield: dict | None = None,
    mu_daily: dict | None = None,
    n_days: int = 252,
    n_paths: int = 20000,
    tau: float = 60.0,
    innov_dist: str = "t",
    seed: int = 42,
    dtype = np.float32,
) -> dict:
    """
    Vectorized correlated-GBM Monte Carlo with GARCH-mean-reverting variance.

    Memory-efficient: stores only one (n_days+1, n_paths, n_assets) array
    of float32 prices (default). For 4 assets / 756 days / 20k paths that is
    ~240 MB, vs. ~2 GB if log-prices and float64 were both kept around.

    Parameters
    ----------
    S0           initial spots                       {ticker: float}
    sigma0       current daily vol per asset         {ticker: float}
    garch_params GARCH(1,1) params per asset         {ticker: {omega, alpha, beta, nu, sigma2_lr}}
    corr_matrix  (n_assets, n_assets) correlation of log-returns
    mode         "risk_neutral" or "real_world"
    risk_free    annualized continuous risk-free rate (used in RN mode)
    div_yield    {ticker: q}, annualized (used in RN mode)
    mu_daily     {ticker: mu}, daily log-return drift (used in RW mode)
    n_days       number of trading days simulated
    n_paths      number of paths
    tau          variance mean-reversion time constant in days
    innov_dist   "t" or "normal"
    seed         RNG seed
    dtype        np.float32 (default, half memory) or np.float64

    Returns
    -------
    dict with keys:
        prices       {ticker: (n_days+1, n_paths) ndarray}
        sigma_daily  {ticker: (n_days+1,) ndarray}   -- deterministic forward vol
        meta         dict (mode, n_days, n_paths, drift_per_day per asset)
    """
    if mode not in {"risk_neutral", "real_world"}:
        raise ValueError(f"Unknown mode: {mode}")

    rng       = np.random.default_rng(seed)
    tickers   = list(S0.keys())
    n_assets  = len(tickers)
    dt        = 1.0 / TRADING_DAYS

    # -------- forward variance (deterministic, shared across paths) ----------
    days_idx = np.arange(1, n_days + 1)
    sigma2_path = np.zeros((n_days + 1, n_assets), dtype=np.float64)
    for i, t in enumerate(tickers):
        s2_lr = garch_params[t]["sigma2_lr"]
        s2_0  = sigma0[t] ** 2
        sigma2_path[0, i]  = s2_0
        sigma2_path[1:, i] = s2_lr + (s2_0 - s2_lr) * np.exp(-days_idx / tau)
    sigma2_path = np.clip(sigma2_path, 1e-10, 0.05)
    sigma_path  = np.sqrt(sigma2_path)

    # -------- standardized correlated innovations ----------------------------
    L = np.linalg.cholesky(corr_matrix).astype(dtype)

    # Allocate the output prices array up-front (largest object).
    prices_arr = np.empty((n_days + 1, n_paths, n_assets), dtype=dtype)
    log_S0_v = np.array([np.log(S0[t]) for t in tickers], dtype=dtype)
    prices_arr[0] = np.exp(log_S0_v)[None, :]

    # Drift matrix per asset per day
    drift_per_day = np.zeros((n_days, n_assets), dtype=dtype)
    if mode == "risk_neutral":
        if div_yield is None:
            div_yield = {t: 0.0 for t in tickers}
        for i, t in enumerate(tickers):
            drift_per_day[:, i] = (risk_free - div_yield[t]
                                   - 0.5 * sigma2_path[1:, i]) * dt
        diff_scale = (sigma_path[1:] * np.sqrt(dt)).astype(dtype)
    else:
        if mu_daily is None:
            raise ValueError("real_world mode requires mu_daily")
        for i, t in enumerate(tickers):
            drift_per_day[:, i] = mu_daily[t]
        diff_scale = sigma_path[1:].astype(dtype)

    # Step-by-step accumulation in log-space, but storing prices.
    # Innovations are drawn one step at a time to avoid keeping the full
    # (n_days, n_paths, n_assets) shock cube in memory.
    log_state = np.broadcast_to(log_S0_v, (n_paths, n_assets)).astype(dtype).copy()

    for d in range(n_days):
        if innov_dist == "t":
            Z_step = np.empty((n_paths, n_assets), dtype=dtype)
            for i, t in enumerate(tickers):
                nu = max(garch_params[t]["nu"], 4.01)
                z  = rng.standard_t(df=nu, size=n_paths).astype(dtype)
                Z_step[:, i] = z / np.float32(np.sqrt(nu / (nu - 2)))
        else:
            Z_step = rng.standard_normal((n_paths, n_assets)).astype(dtype)

        Z_corr = Z_step @ L.T                                  # (n_paths, n_assets)
        log_state = log_state + drift_per_day[d][None, :] + diff_scale[d][None, :] * Z_corr
        prices_arr[d + 1] = np.exp(log_state)

    # Repackage to {ticker: (n_days+1, n_paths)} for downstream code.
    prices_d = {t: prices_arr[:, :, i] for i, t in enumerate(tickers)}
    sigma_d  = {t: sigma_path[:, i]    for i, t in enumerate(tickers)}

    return {
        "prices":      prices_d,
        "sigma_daily": sigma_d,
        "meta": {
            "mode": mode, "n_days": n_days, "n_paths": n_paths,
            "tau":  tau,  "innov_dist": innov_dist, "seed": seed,
            "dtype": str(np.dtype(dtype)),
            "drift_per_day": {t: float(drift_per_day[:, i].mean())
                              for i, t in enumerate(tickers)},
        },
    }


def diagnostic_table(sim: dict, S0: dict,
                     horizons=(5, 21, 63, 126, 252, 504, 756)) -> pd.DataFrame:
    """Distribution snapshot at several horizons."""
    rows = []
    for t, P in sim["prices"].items():
        n_steps = P.shape[0] - 1
        for h in horizons:
            if h > n_steps:
                continue
            Pt = P[h]
            rows.append({
                "Ticker": t, "Days": h, "S0": S0[t],
                "Mean":   float(Pt.mean()),  "Median": float(np.median(Pt)),
                "P5":     float(np.percentile(Pt, 5)),
                "P95":    float(np.percentile(Pt, 95)),
                "Ret_med_%": (float(np.median(Pt)) / S0[t] - 1) * 100,
                "Vol_ann":   float(sim["sigma_daily"][t][h] * np.sqrt(TRADING_DAYS)),
            })
    return pd.DataFrame(rows)


def martingale_check(sim: dict, S0: dict, r: float, q: dict) -> pd.DataFrame:
    """For RN mode: E[S_T * exp(-(r-q)T)] should equal S_0."""
    rows = []
    n_steps = next(iter(sim["prices"].values())).shape[0] - 1
    T = n_steps / TRADING_DAYS
    for t, P in sim["prices"].items():
        ST   = P[-1].astype(np.float64)
        disc = np.exp(-(r - q[t]) * T)
        lhs  = float((ST * disc).mean())
        se   = float((ST * disc).std() / np.sqrt(len(ST)))
        rows.append({"Ticker": t, "S0": S0[t], "E[disc·ST]": lhs,
                     "ratio": lhs / S0[t], "MC_SE": se,
                     "OK": abs(lhs / S0[t] - 1) < 0.01})
    return pd.DataFrame(rows)


## 7. Certificate Payoff Engine

Fully vectorized worst-of barrier payoff supporting:
- conditional periodic coupons
- optional **memory** (missed coupons accrue)
- optional **autocall** (early redemption at par + paid coupons)
- **capital protection** at maturity (0 = pure barrier, 1 = full protection)

The engine returns per-path P&L plus monitoring arrays useful for risk
analysis (drawdowns, barrier breaches, time-to-autocall).

In [ ]:
# %% CERTIFICATE PAYOFF
def price_certificate(sim: dict, spec: CertificateSpec,
                      *, discount_rate: float | None = None) -> dict:
    """
    Vectorized worst-of barrier certificate payoff.

    Single pass over coupon observation dates; tracks:
      - per-path cumulative coupons (with optional memory)
      - autocall state, redemption day
      - discounted PV (when `discount_rate` provided, typically RN runs)

    Parameters
    ----------
    sim            output of `simulate_paths`
    spec           CertificateSpec
    discount_rate  if provided, also computes the present value (each cashflow
                   discounted at its own date)

    Returns
    -------
    dict with:
      summary       aggregate metrics
      path_df       per-path P&L, capital, coupons, autocall flag, max DD…
      worst_ratio   (T+1, n_paths) min over assets of S/S0
      drawdowns     (T+1, n_paths) running drawdown of the worst-of
      coupon_days   array of coupon observation days
    """
    tickers   = list(sim["prices"].keys())
    n_assets  = len(tickers)
    P = np.stack([sim["prices"][t] for t in tickers], axis=0)  # (A, T+1, N)
    n_paths = P.shape[2]
    n_steps_total = P.shape[1] - 1

    if spec.maturity_days > n_steps_total:
        raise ValueError(f"maturity_days ({spec.maturity_days}) > simulated horizon ({n_steps_total})")

    # truncate to maturity
    P    = P[:, : spec.maturity_days + 1, :]
    S0_v = P[:, 0, 0].astype(np.float64)                       # (A,) initial spots

    # ------------------ schedules ------------------------------------------
    coupon_days = np.arange(spec.coupon_freq_days,
                            spec.maturity_days + 1,
                            spec.coupon_freq_days)
    if coupon_days.size == 0 or coupon_days[-1] != spec.maturity_days:
        coupon_days = np.append(coupon_days, spec.maturity_days)

    if spec.autocall:
        ac_set = set(np.arange(spec.autocall_first_day,
                               spec.maturity_days,
                               spec.autocall_freq_days).tolist())
    else:
        ac_set = set()

    # ------------------ worst-of ratio path --------------------------------
    ratios       = (P / S0_v[:, None, None]).astype(np.float64)
    worst_ratio  = ratios.min(axis=0)                          # (T+1, N)

    # ------------------ coupon engine --------------------------------------
    coupon_per_period = (spec.coupon_annual_pct / 100.0) * spec.notional \
                        * (spec.coupon_freq_days / TRADING_DAYS)

    coupons_undisc   = np.zeros(n_paths)
    coupons_disc     = np.zeros(n_paths)   # only filled if discount_rate is not None
    pending          = np.zeros(n_paths)   # for memory effect
    autocalled       = np.zeros(n_paths, dtype=bool)
    redemption_day   = np.full(n_paths, spec.maturity_days, dtype=np.int32)
    barrier_hits     = np.zeros(n_paths, dtype=np.int32)
    do_disc          = discount_rate is not None

    barrier_breached_anytime = (worst_ratio < spec.barrier_pct).any(axis=0)

    for d in coupon_days:
        d = int(d)
        alive = ~autocalled
        wr_d  = worst_ratio[d]
        pays  = alive & (wr_d >= spec.barrier_pct)

        if spec.memory_coupon:
            pay_amt = coupon_per_period + pending
        else:
            pay_amt = np.full(n_paths, coupon_per_period)

        # accumulate paid coupons
        coupons_undisc[pays] += pay_amt[pays]
        if do_disc:
            df = np.exp(-discount_rate * d / TRADING_DAYS)
            coupons_disc[pays] += df * pay_amt[pays]

        # update memory bucket
        if spec.memory_coupon:
            pending[pays] = 0.0
            pending[alive & ~pays] += coupon_per_period

        # barrier breach counter
        barrier_hits += (alive & (wr_d < spec.barrier_pct)).astype(np.int32)

        # autocall observation (if d in autocall schedule and < maturity)
        if d in ac_set:
            trig = alive & (wr_d >= spec.autocall_trigger_pct)
            autocalled[trig]     = True
            redemption_day[trig] = d

    # ------------------ capital at redemption ------------------------------
    final_wr = worst_ratio[spec.maturity_days]
    above    = final_wr >= spec.barrier_pct

    barrier_capital = np.where(above, spec.notional,
                                       spec.notional * final_wr)
    floor = spec.capital_protection_pct * spec.notional
    barrier_capital = np.maximum(barrier_capital, floor)

    capital = np.where(autocalled, spec.notional, barrier_capital)

    # ------------------ totals + P&L ---------------------------------------
    total_payoff = capital + coupons_undisc
    pnl          = total_payoff - spec.issue_price

    if do_disc:
        T_yrs   = redemption_day / TRADING_DAYS
        df_cap  = np.exp(-discount_rate * T_yrs)
        pv      = capital * df_cap + coupons_disc
    else:
        pv = np.full(n_paths, np.nan)

    # ------------------ drawdown of worst-of -------------------------------
    cummax    = np.maximum.accumulate(worst_ratio, axis=0)
    drawdowns = 1.0 - worst_ratio / cummax
    max_dd    = drawdowns.max(axis=0)

    # ------------------ summary --------------------------------------------
    losses_terminal = 1 - final_wr
    pnl_loss        = -pnl / spec.issue_price

    summary = {
        "fair_value":            float(pv.mean())     if do_disc else float("nan"),
        "fv_std_error":          float(pv.std() / np.sqrt(n_paths)) if do_disc else float("nan"),
        "expected_payoff":       float(total_payoff.mean()),
        "expected_pnl":          float(pnl.mean()),
        "median_payoff":         float(np.median(total_payoff)),
        "prob_profit":           float((pnl > 0).mean()),
        "prob_loss":             float((pnl < 0).mean()),
        "prob_breach_barrier":   float(barrier_breached_anytime.mean()),
        "prob_autocall":         float(autocalled.mean()),
        "expected_coupons":      float(coupons_undisc.mean()),
        "median_coupons":        float(np.median(coupons_undisc)),
        "expected_capital":      float(capital.mean()),
        "max_drawdown_mean":     float(max_dd.mean()),
        "max_drawdown_p95":      float(np.percentile(max_dd, 95)),
    }
    for q in (90, 95, 99):
        v = np.percentile(losses_terminal, q)
        e = losses_terminal[losses_terminal >= v].mean()
        summary[f"VaR_{q}_worstof"] = float(v)
        summary[f"ES_{q}_worstof"]  = float(e)
        v = np.percentile(pnl_loss, q)
        e = pnl_loss[pnl_loss >= v].mean()
        summary[f"VaR_{q}_pnl"] = float(v)
        summary[f"ES_{q}_pnl"]  = float(e)

    path_df = pd.DataFrame({
        "capital":           capital,
        "coupons":           coupons_undisc,
        "total_payoff":      total_payoff,
        "pnl":               pnl,
        "pv":                pv,
        "barrier_hit_count": barrier_hits,
        "autocalled":        autocalled,
        "redemption_day":    redemption_day,
        "max_drawdown":      max_dd,
        "final_worst_ratio": final_wr,
    })

    return {
        "summary":     summary,
        "path_df":     path_df,
        "worst_ratio": worst_ratio,
        "drawdowns":   drawdowns,
        "coupon_days": coupon_days,
    }


## 8. Risk-Neutral Pricing (Fair Value)

Run the engine in `risk_neutral` mode and price the certificate.

In [ ]:
# %% RISK NEUTRAL PRICING
S0 = {t: float(prices_df[t].iloc[-1]) for t in TICKERS}

# correlation as numpy in TICKERS order
corr_np = corr_matrix.loc[TICKERS, TICKERS].values

sim_rn = simulate_paths(
    S0=S0, sigma0=sigma0,
    garch_params=garch_params,
    corr_matrix=corr_np,
    mode="risk_neutral",
    risk_free=RISK_FREE_RATE,
    div_yield=DIVIDEND_YIELD,
    n_days=N_DAYS, n_paths=N_PATHS,
    tau=TAU, innov_dist=INNOV_DIST, seed=SEED,
)
print(f"Risk-neutral simulation OK — shape {sim_rn['prices'][TICKERS[0]].shape}")


In [ ]:
# %% RISK NEUTRAL - martingale check & diagnostic
mg = martingale_check(sim_rn, S0, RISK_FREE_RATE, DIVIDEND_YIELD)
print("Martingale check (RN):")
print(mg.round(5).to_string(index=False))

diag_rn = diagnostic_table(sim_rn, S0)
print("\nDistribution snapshot:")
diag_rn.round(3).head(20)


In [ ]:
# %% RISK NEUTRAL - certificate fair value
res_rn = price_certificate(sim_rn, CERTIFICATE, discount_rate=RISK_FREE_RATE)
fv     = res_rn["summary"]["fair_value"]
fv_se  = res_rn["summary"]["fv_std_error"]
print(f"Fair Value (RN, discounted): {fv:.4f}   (MC std error ≈ {fv_se:.4f})")
print(f"95% MC CI: [{fv - 1.96 * fv_se:.4f},  {fv + 1.96 * fv_se:.4f}]")

print("\nKey RN summary:")
print(pd.Series(res_rn["summary"]).round(4))


## 9. Real-World Risk Analysis

Run the engine in `real_world` mode (drift = historical μ) for risk metrics
and scenario analysis.

In [ ]:
# %% REAL WORLD - simulation with historical drift
mu_daily = {t: float(log_ret[f"{t}_rD"].mean()) for t in TICKERS}
print("Historical daily log-drift per asset:")
for t in TICKERS:
    ann = mu_daily[t] * TRADING_DAYS
    print(f"   {t}: {mu_daily[t]:.6f}  (annualized ≈ {ann:.2%})")

sim_rw = simulate_paths(
    S0=S0, sigma0=sigma0,
    garch_params=garch_params,
    corr_matrix=corr_np,
    mode="real_world",
    mu_daily=mu_daily,
    n_days=N_DAYS, n_paths=N_PATHS,
    tau=TAU, innov_dist=INNOV_DIST, seed=SEED,
)
print(f"\nReal-world simulation OK — shape {sim_rw['prices'][TICKERS[0]].shape}")

diag_rw = diagnostic_table(sim_rw, S0)
diag_rw.round(3).head(20)


In [ ]:
# %% REAL WORLD - certificate risk metrics
res_rw = price_certificate(sim_rw, CERTIFICATE, discount_rate=None)

print("Real-world certificate risk metrics:")
print(pd.Series(res_rw["summary"]).round(4))

print("\nP&L distribution percentiles:")
print(res_rw["path_df"]["pnl"].describe(percentiles=[.05, .25, .5, .75, .95]).round(3))


### MC Plots — Simulated Paths, Terminal Distribution, Worst-of, P&L

In [ ]:
# %% MC PLOT - simulated price fan per ticker (RW used here)
sim_to_plot = sim_rw  # change to sim_rn to see risk-neutral

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, t, color in zip(axes.flat, TICKERS, PALETTE):
    P = sim_to_plot["prices"][t]
    days = np.arange(P.shape[0])
    # 50 random paths in light grey
    sample = np.random.choice(P.shape[1], size=min(50, P.shape[1]), replace=False)
    ax.plot(days, P[:, sample], color="grey", alpha=0.15, lw=0.6)
    # percentile envelope
    p5  = np.percentile(P, 5,  axis=1)
    p50 = np.percentile(P, 50, axis=1)
    p95 = np.percentile(P, 95, axis=1)
    ax.plot(days, p50, color=color, lw=1.8, label="median")
    ax.plot(days, p5,  color=color, lw=1.0, ls="--", label="5%")
    ax.plot(days, p95, color=color, lw=1.0, ls="--", label="95%")
    ax.set_title(f"{t} — Simulated paths")
    ax.set_xlabel("trading days")
    ax.set_ylabel("price")
    ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


In [ ]:
# %% MC PLOT - terminal distribution + worst-of evolution
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Terminal distribution (RW) for each underlying
for i, t in enumerate(TICKERS):
    P_T = sim_rw["prices"][t][-1]
    axes[0].hist(P_T / S0[t] - 1, bins=80, alpha=0.45,
                 color=PALETTE[i], label=t, density=True)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("Real-World Terminal Return Distribution per Underlying")
axes[0].set_xlabel("terminal return")
axes[0].legend()

# Worst-of evolution percentiles
wr = res_rw["worst_ratio"]
days = np.arange(wr.shape[0])
axes[1].plot(days, np.percentile(wr, 5,  axis=1), "r--", lw=1.2, label="5%")
axes[1].plot(days, np.percentile(wr, 50, axis=1), "k-",  lw=1.6, label="median")
axes[1].plot(days, np.percentile(wr, 95, axis=1), "g--", lw=1.2, label="95%")
axes[1].axhline(CERTIFICATE.barrier_pct, color="red", lw=1.5, ls=":",
                label=f"barrier {CERTIFICATE.barrier_pct:.0%}")
axes[1].set_title("Worst-Of Ratio — Path Envelope")
axes[1].set_xlabel("trading days")
axes[1].set_ylabel("min over assets of S/S0")
axes[1].legend()

plt.tight_layout(); plt.show()


In [ ]:
# %% MC PLOT - certificate payoff distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.hist(res_rw["path_df"]["pnl"], bins=80, color="#1f77b4", alpha=0.8)
for q, c in [(5, "red"), (50, "black"), (95, "green")]:
    v = np.percentile(res_rw["path_df"]["pnl"], q)
    ax.axvline(v, ls="--", color=c, lw=1, label=f"{q}%: {v:.1f}")
ax.set_title("Total P&L Distribution (RW)")
ax.set_xlabel("P&L"); ax.legend(fontsize=8)

ax = axes[1]
ax.hist(res_rw["path_df"]["coupons"], bins=80, color="#2ca02c", alpha=0.8)
ax.set_title("Total Coupons Received")
ax.set_xlabel("coupons")

ax = axes[2]
ax.hist(res_rw["path_df"]["max_drawdown"], bins=80, color="#d62728", alpha=0.8)
ax.axvline(1 - CERTIFICATE.barrier_pct, color="black", ls="--", lw=1.2,
           label=f"barrier DD = {1-CERTIFICATE.barrier_pct:.0%}")
ax.set_title("Worst-Of Max Drawdown")
ax.set_xlabel("drawdown"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


## 10. Reporting & Excel Export

A single workbook gathers every relevant table:
- input configuration & certificate spec
- historical performance/risk dashboard
- correlation matrix
- GARCH parameters
- diagnostic snapshots (RN & RW)
- martingale check (RN)
- certificate fair value (RN) and risk metrics (RW)
- P&L distribution percentiles

In [ ]:
# %% EXPORT - build workbook
def _spec_dict(spec: CertificateSpec) -> dict:
    return {f.name: getattr(spec, f.name) for f in spec.__dataclass_fields__.values()}


def percentile_table(series: pd.Series, qs=(1, 5, 25, 50, 75, 95, 99)) -> pd.DataFrame:
    return pd.DataFrame({"percentile": [f"{q}%" for q in qs],
                         "value":      [np.percentile(series, q) for q in qs]})


config_df = pd.DataFrame.from_dict({
    "tickers":          ", ".join(TICKERS),
    "n_days":           N_DAYS,
    "n_paths":          N_PATHS,
    "tau":              TAU,
    "innov_dist":       INNOV_DIST,
    "risk_free_rate":   RISK_FREE_RATE,
    "rolling_window":   ROLLING_WIN,
    "trading_days":     TRADING_DAYS,
    "seed":             SEED,
}, orient="index", columns=["value"])

div_df  = pd.DataFrame.from_dict(DIVIDEND_YIELD, orient="index", columns=["dividend_yield"])
spec_df = pd.DataFrame.from_dict(_spec_dict(CERTIFICATE), orient="index", columns=["value"])

with pd.ExcelWriter(OUTPUT_FILE, engine="xlsxwriter") as xw:
    config_df.to_excel(xw, sheet_name="01_Config")
    div_df.to_excel  (xw, sheet_name="01_Config", startrow=len(config_df) + 2)
    spec_df.to_excel (xw, sheet_name="02_Certificate")

    prices_df.to_excel(xw, sheet_name="03_Prices_aligned")
    log_ret.to_excel  (xw, sheet_name="04_LogReturns")
    rolling_vol.to_excel(xw, sheet_name="05_RollingVol")
    garch_vol_history.to_excel(xw, sheet_name="06_GARCH_Vol")

    g_summary.to_excel(xw, sheet_name="07_GARCH_Params")
    hist_summary_df.to_excel(xw, sheet_name="08_Hist_Risk_Perf")
    corr_matrix.to_excel(xw, sheet_name="09_Correlation")

    diag_rn.to_excel(xw, sheet_name="10_RN_Diagnostic", index=False)
    mg.to_excel     (xw, sheet_name="11_RN_Martingale", index=False)
    pd.Series(res_rn["summary"], name="value").to_frame().to_excel(
        xw, sheet_name="12_RN_FairValue")

    diag_rw.to_excel(xw, sheet_name="13_RW_Diagnostic", index=False)
    pd.Series(res_rw["summary"], name="value").to_frame().to_excel(
        xw, sheet_name="14_RW_RiskMetrics")
    percentile_table(res_rw["path_df"]["pnl"]).to_excel(
        xw, sheet_name="15_RW_PnL_Percentiles", index=False)
    percentile_table(res_rw["path_df"]["coupons"]).to_excel(
        xw, sheet_name="16_RW_Coupon_Percentiles", index=False)

print(f"Report written to: {OUTPUT_FILE}")


In [ ]:
# %% FINAL - one-line console summary
print(f"""
========================================================================
  CERTIFICATE  ·  Worst-Of Barrier  ·  {N_PATHS:,} paths × {N_DAYS} days
------------------------------------------------------------------------
  Underlyings           : {", ".join(TICKERS)}
  Risk-free rate        : {RISK_FREE_RATE:.4f}
  Barrier               : {CERTIFICATE.barrier_pct:.0%}
  Coupon (annual)       : {CERTIFICATE.coupon_annual_pct:.2f}%
  Maturity              : {CERTIFICATE.maturity_days} days
  Memory / Autocall     : {CERTIFICATE.memory_coupon} / {CERTIFICATE.autocall}
------------------------------------------------------------------------
  Fair Value (RN)       : {res_rn['summary']['fair_value']:.4f}
                          (issue price = {CERTIFICATE.issue_price:.2f})
  Prob. profit (RW)     : {res_rw['summary']['prob_profit']:.2%}
  Prob. barrier breach  : {res_rw['summary']['prob_breach_barrier']:.2%}
  Prob. autocall        : {res_rw['summary']['prob_autocall']:.2%}
  Expected coupons (RW) : {res_rw['summary']['expected_coupons']:.2f}
  VaR 95% / ES 95% (PnL): {res_rw['summary']['VaR_95_pnl']:.2%} / {res_rw['summary']['ES_95_pnl']:.2%}
========================================================================
""")
